# 01 - SmartCentres Scraping

This notebook collects SmartCentres data

Final CSV outputs:
- `data/smartcentres_financials_from_pdfs.csv`
- `data/smartcentres_properties.csv`
- `data/smartcentres_distributions.csv`

PDF outputs:
- `pdfs/` contains the official SmartCentres PDFs used for extraction.

In [11]:
# Install only the basic libraries needed for this SmartCentres source.
# requests opens public PDF/page URLs.
# BeautifulSoup reads the visible distributions table.
# pandas creates DataFrames and CSV files.
# pdfplumber extracts text from the selected official PDFs.
!pip -q install requests beautifulsoup4 lxml pandas pdfplumber

In [12]:
import re
import time
from pathlib import Path

import pandas as pd
import pdfplumber
import requests
from bs4 import BeautifulSoup
try:
    from IPython.display import display
except ImportError:
    # Local validation fallback. Colab has display(), but plain Python may not.
    def display(value):
        print(value)


DATA_DIR = Path("/content/data")
PDF_DIR = Path("/content/pdfs")
DATA_DIR.mkdir(parents=True, exist_ok=True)
PDF_DIR.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (MBA assignment basic scraper)"})

# On some local Windows/Anaconda setups, public websites can fail because of
# certificate-chain configuration. Colab usually works without this. Setting
# verify=False keeps the notebook runnable in both places for this assignment.
session.verify = False
requests.packages.urllib3.disable_warnings()

## Manually Selected PDFs

Annual reports are used for operating metrics such as occupancy and property count.

Q4 financial statements are used for annual financial rows such as total assets, debt, rental income, net income, operating cash flow, and distributions paid.

Q1/Q2/Q3 statements are skipped because quarterly partial-year figures should not be mixed with annual trends.

In [13]:
SMARTCENTRES_PDFS = [
    {"year": 2022, "kind": "annual_report", "filename": "2022_SmartCentres_REIT_Annual_Report.pdf", "url": "https://smartcentres.com/wp-content/uploads/2023/05/2022-SmartCentres-REIT-Annual-Report.pdf"},
    {"year": 2023, "kind": "annual_report", "filename": "2023_SmartCentres_REIT_Annual_Report.pdf", "url": "https://smartcentres.com/wp-content/uploads/2024/05/2023-SmartCentres-REIT-Annual-Report.pdf"},
    {"year": 2024, "kind": "annual_report", "filename": "2024_SmartCentres_REIT_Annual_Report.pdf", "url": "https://smartcentres.com/wp-content/uploads/2025/04/2024-SmartCentres-REIT-Annual-Report-.pdf"},
    {"year": 2025, "kind": "annual_report", "filename": "2025_SmartCentres_REIT_Annual_Report.pdf", "url": "https://smartcentres.com/wp-content/uploads/2026/04/2025-SmartCentres-REIT-Annual-Report.pdf"},
    {"year": 2022, "kind": "q4_financial_statements", "filename": "2022_Q4_Financial_Statements.pdf", "url": "https://smartcentres.com/wp-content/uploads/2023/02/2022-Q4-Financial-Statements.pdf"},
    {"year": 2023, "kind": "q4_financial_statements", "filename": "2023_Q4_Financial_Statements.pdf", "url": "https://smartcentres.com/wp-content/uploads/2023/05/2023-Q4-Financial-Statements.pdf"},
    {"year": 2024, "kind": "q4_financial_statements", "filename": "2024_Q4_Financial_Statements.pdf", "url": "https://smartcentres.com/wp-content/uploads/2024/05/Q4-2024-Financial-Statements.pdf"},
    {"year": 2025, "kind": "q4_financial_statements", "filename": "2025_Q4_Financial_Statements.pdf", "url": "https://smartcentres.com/wp-content/uploads/2026/02/Q4-2025-Financial-Statements.pdf"},
]

display(pd.DataFrame(SMARTCENTRES_PDFS))

,year,kind,filename,url
0,2022,annual_report,2022_SmartCentres_REIT_Annual_Report.pdf,https://smartcentres.com/wp-content/uploads/20...
1,2023,annual_report,2023_SmartCentres_REIT_Annual_Report.pdf,https://smartcentres.com/wp-content/uploads/20...
2,2024,annual_report,2024_SmartCentres_REIT_Annual_Report.pdf,https://smartcentres.com/wp-content/uploads/20...
3,2025,annual_report,2025_SmartCentres_REIT_Annual_Report.pdf,https://smartcentres.com/wp-content/uploads/20...
4,2022,q4_financial_statements,2022_Q4_Financial_Statements.pdf,https://smartcentres.com/wp-content/uploads/20...
5,2023,q4_financial_statements,2023_Q4_Financial_Statements.pdf,https://smartcentres.com/wp-content/uploads/20...
6,2024,q4_financial_statements,2024_Q4_Financial_Statements.pdf,https://smartcentres.com/wp-content/uploads/20...
7,2025,q4_financial_statements,2025_Q4_Financial_Statements.pdf,https://smartcentres.com/wp-content/uploads/20...


In [14]:
def download_pdf(pdf_info):
    """Download one selected official PDF into the pdfs folder."""
    output_path = PDF_DIR / pdf_info["filename"]
    if output_path.exists() and output_path.stat().st_size > 0:
        return output_path
    response = session.get(pdf_info["url"], timeout=60)
    response.raise_for_status()
    output_path.write_bytes(response.content)
    time.sleep(0.5)
    return output_path

for pdf_info in SMARTCENTRES_PDFS:
    pdf_info["local_path"] = str(download_pdf(pdf_info))

display(pd.DataFrame(SMARTCENTRES_PDFS)[["year", "kind", "filename"]])

,year,kind,filename
0,2022,annual_report,2022_SmartCentres_REIT_Annual_Report.pdf
1,2023,annual_report,2023_SmartCentres_REIT_Annual_Report.pdf
2,2024,annual_report,2024_SmartCentres_REIT_Annual_Report.pdf
3,2025,annual_report,2025_SmartCentres_REIT_Annual_Report.pdf
4,2022,q4_financial_statements,2022_Q4_Financial_Statements.pdf
5,2023,q4_financial_statements,2023_Q4_Financial_Statements.pdf
6,2024,q4_financial_statements,2024_Q4_Financial_Statements.pdf
7,2025,q4_financial_statements,2025_Q4_Financial_Statements.pdf


## Targeted PDF Extraction

The line items below were chosen after manual review of the reports. The code is not trying to discover what matters. It simply extracts selected rows from selected tables.

PDF text still needs small parsing helpers because a visible table in a PDF is not a clean spreadsheet table internally.

In [15]:
def clean_text(value):
    return re.sub(r"\s+", " ", str(value or "")).strip()

def value_to_number(value):
    text = str(value or "").strip()
    if not text:
        return None
    is_negative = "(" in text and ")" in text
    text = text.replace("$", "").replace(",", "").replace("%", "").replace("X", "").replace("x", "")
    text = text.replace("(", "").replace(")", "")
    match = re.search(r"-?\d+(\.\d+)?", text)
    if not match:
        return None
    number = float(match.group(0))
    return -number if is_negative and number > 0 else number

def extract_numbers_from_line(line):
    pattern = r"\(?\$?\d{1,3}(?:,\d{3})*(?:\.\d+)?\)?\s*(?:%|X|x)?"
    return [clean_text(value) for value in re.findall(pattern, line) if clean_text(value)]

def line_starts_with_metric(line, metric):
    line_lower = clean_text(line).lower()
    metric_lower = metric.lower()
    if not line_lower.startswith(metric_lower):
        return False
    remainder = line_lower[len(metric_lower):].strip()
    return bool(remainder) and not remainder.startswith("and ")

def row_for_value(source_pdf, source_page, table_name, metric, period, value_raw, unit_or_note, note):
    return {
        "source_pdf": source_pdf,
        "source_page": source_page,
        "table_name": table_name,
        "metric_or_line_item": metric,
        "period": period,
        "value_raw": value_raw,
        "value_numeric": value_to_number(value_raw),
        "unit_or_note": unit_or_note,
        "extraction_note": note,
    }

def dedupe_rows(rows):
    unique = {}
    for row in rows:
        key = (row["source_pdf"], row["table_name"], row["metric_or_line_item"], row["period"], row["value_raw"])
        unique.setdefault(key, row)
    return list(unique.values())

In [16]:
KEY_METRICS = {
    "Retail properties": "count",
    "Office properties": "count",
    "Self-storage properties": "count",
    "Residential properties": "count",
    "Industrial properties": "count",
    "Properties under development": "count",
    "Total number of properties with an ownership interest": "count",
    "Gross leasable retail, office and industrial area": "thousands of sq. ft.",
    "In-place and committed occupancy rate": "percent",
    "Average lease term to maturity": "years",
    "In-place net retail rental rate excluding Anchors": "dollars per occupied sq. ft.",
    "Investment properties": "thousands of Canadian dollars",
    "Total unencumbered assets": "thousands of Canadian dollars",
    "Total assets": "thousands of Canadian dollars",
    "NAV per Unit": "dollars per unit",
    "Debt to Aggregate Assets": "percent",
    "Adjusted Debt to Adjusted EBITDA": "times",
    "Weighted average interest rate": "percent",
    "Weighted average term of debt": "years",
    "Interest coverage ratio": "times",
    "Units outstanding": "units",
}

FINANCIAL_ROWS = {
    "CONSOLIDATED BALANCE SHEETS": {
        "Investment properties": "thousands of Canadian dollars",
        "Cash and cash equivalents": "thousands of Canadian dollars",
        "Total assets": "thousands of Canadian dollars",
        "Debt": "thousands of Canadian dollars",
        "Total liabilities": "thousands of Canadian dollars",
        "Total equity": "thousands of Canadian dollars",
    },
    "CONSOLIDATED STATEMENTS OF INCOME AND COMPREHENSIVE INCOME": {
        "Rentals from investment properties and other": "thousands of Canadian dollars",
        "Property operating costs and other": "thousands of Canadian dollars",
        "Net rental income and other": "thousands of Canadian dollars",
        "General and administrative expense": "thousands of Canadian dollars",
        "Interest expense": "thousands of Canadian dollars",
        "Interest income": "thousands of Canadian dollars",
        "Net income and comprehensive income": "thousands of Canadian dollars",
    },
    "CONSOLIDATED STATEMENTS OF CASH FLOWS": {
        "Cash flows provided by operating activities": "thousands of Canadian dollars",
        "Cash flows used in financing activities": "thousands of Canadian dollars",
        "Distributions paid on Trust Units": "thousands of Canadian dollars",
        "Cash flows used in investing activities": "thousands of Canadian dollars",
    },
}

In [17]:
def extract_key_metrics_from_annual_report(pdf_info):
    pdf_path = Path(pdf_info["local_path"])
    report_year = int(pdf_info["year"])
    periods = [str(report_year), str(report_year - 1), str(report_year - 2)]
    rows = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_index, page in enumerate(pdf.pages[:25], start=1):
            text = page.extract_text(x_tolerance=1, y_tolerance=3) or ""
            if "Key Operational" not in text and "Portfolio Information" not in text:
                continue
            for raw_line in text.split("\n"):
                line = clean_text(raw_line)
                for metric, unit in KEY_METRICS.items():
                    if line_starts_with_metric(line, metric):
                        values = extract_numbers_from_line(line)[-len(periods):]
                        for period, value_raw in zip(periods[:len(values)], values):
                            rows.append(row_for_value(pdf_path.name, page_index, "Key Operational, Development and Financial Information", metric, period, value_raw, unit, "Targeted extraction from manually selected annual report table"))
    return rows

def statement_name_for_page(text):
    text_upper = text.upper()
    for statement_name in FINANCIAL_ROWS:
        if statement_name in text_upper:
            return statement_name
    return None

def extract_financial_rows_from_q4_statement(pdf_info):
    pdf_path = Path(pdf_info["local_path"])
    report_year = int(pdf_info["year"])
    periods = [str(report_year), str(report_year - 1)]
    rows = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_index, page in enumerate(pdf.pages, start=1):
            text = page.extract_text(x_tolerance=1, y_tolerance=3) or ""
            statement_name = statement_name_for_page(text)
            if statement_name is None:
                continue
            for raw_line in text.split("\n"):
                line = clean_text(raw_line)
                for metric, unit in FINANCIAL_ROWS[statement_name].items():
                    if line_starts_with_metric(line, metric):
                        values = extract_numbers_from_line(line)[-len(periods):]
                        for period, value_raw in zip(periods[:len(values)], values):
                            rows.append(row_for_value(pdf_path.name, page_index, statement_name.title(), metric, period, value_raw, unit, "Targeted extraction from manually selected Q4 financial statement"))
    return rows

In [ ]:
key_metric_rows = []
financial_rows = []
for pdf_info in SMARTCENTRES_PDFS:
    if pdf_info["kind"] == "annual_report":
        key_metric_rows.extend(extract_key_metrics_from_annual_report(pdf_info))
    elif pdf_info["kind"] == "q4_financial_statements":
        financial_rows.extend(extract_financial_rows_from_q4_statement(pdf_info))

key_metrics_df = pd.DataFrame(dedupe_rows(key_metric_rows))
financials_df = pd.DataFrame(dedupe_rows(financial_rows))

property_metric_names = [
    "Retail properties", "Office properties", "Self-storage properties",
    "Residential properties", "Industrial properties", "Properties under development",
    "Total number of properties with an ownership interest",
    "Gross leasable retail, office and industrial area",
    "In-place and committed occupancy rate",
    "Average lease term to maturity",
    "In-place net retail rental rate excluding Anchors",
]
properties_df = key_metrics_df[key_metrics_df["metric_or_line_item"].isin(property_metric_names)].copy()
annual_financial_metrics_df = key_metrics_df[~key_metrics_df["metric_or_line_item"].isin(property_metric_names)].copy()
smartcentres_financials_df = pd.concat([financials_df, annual_financial_metrics_df], ignore_index=True)

smartcentres_financials_df.to_csv(DATA_DIR / "smartcentres_financials_from_pdfs.csv", index=False)
properties_df.to_csv(DATA_DIR / "smartcentres_properties.csv", index=False)
print("Saved financial rows:", len(smartcentres_financials_df))
print("Saved property rows:", len(properties_df))
display(smartcentres_financials_df.head())
display(properties_df.head())

## SmartCentres Distribution Table

A REIT usually calls its cash payments to investors `distributions`. For this assignment, they are similar to dividends.

The SmartCentres website has a visible distribution table. This notebook keeps it as the official company-source distribution CSV.

In [ ]:
response = session.get("https://smartcentres.com/investing/", timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")
page_text = soup.get_text(" ", strip=True)

pattern = re.compile(
    r"(?P<period>[A-Za-z]+)\s+"
    r"(?P<record_date>[A-Za-z]+\s+\d{1,2},\s+\d{4})\s+"
    r"(?P<payment_date>[A-Za-z]+\s+\d{1,2},\s+\d{4})\s+"
    r"\$\s*(?P<dividend_amount>\d+(\.\d+)?)"
)

distribution_rows = []
for match in pattern.finditer(page_text):
    distribution_rows.append({
        "ticker": "SRU-UN.TO",
        "source": "SmartCentres investor website",
        "period": match.group("period"),
        "record_date": match.group("record_date"),
        "payment_date": match.group("payment_date"),
        "dividend_amount": float(match.group("dividend_amount")),
        "notes": "Official visible SmartCentres distribution table",
    })

smartcentres_distributions_df = pd.DataFrame(distribution_rows)
smartcentres_distributions_df.to_csv(DATA_DIR / "smartcentres_distributions.csv", index=False)
print("Saved distributions:", len(smartcentres_distributions_df))
display(smartcentres_distributions_df)

In [ ]:
for path in [
    DATA_DIR / "smartcentres_financials_from_pdfs.csv",
    DATA_DIR / "smartcentres_properties.csv",
    DATA_DIR / "smartcentres_distributions.csv",
]:
    frame = pd.read_csv(path)
    print(path.name, "rows:", len(frame), "columns:", list(frame.columns))